# LeetCode 3193 — Count the Number of Inversions bằng Python

- Link: https://leetcode.com/problems/count-the-number-of-inversions/
- Độ khó: **Hard**
- Chủ đề: **DP đếm hoán vị theo số nghịch thế** — phần tử thứ `i` (theo thứ tự tương đối trong tiền tố) tạo thêm `0..i` nghịch thế; ràng buộc tiền tố ⇒ "lọc" trạng thái; tối ưu chuyển trạng thái bằng **tổng tiền tố (cửa sổ trượt)**

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def numberOfPermutations(self, n: int, requirements: List[List[int]]) -> int:
```

## 1. Mục tiêu học

1. Số nghịch thế của tiền tố `perm[0..i]` chỉ phụ thuộc **thứ tự tương đối** của các phần tử trong tiền tố.
2. Thêm phần tử thứ `i` (chỉ số 0) vào cuối: nếu nó là phần tử lớn thứ `t + 1` từ trên xuống trong `i + 1` phần tử thì tạo thêm đúng `t ∈ [0, i]` nghịch thế, và mọi `t` có đúng một cách (song ánh với **mã Lehmer**).
3. DP `dp[i][j]` = số thứ tự tương đối của tiền tố độ dài `i + 1` có `j` nghịch thế; ràng buộc `[end, cnt]` ⇒ giữ lại chỉ `dp[end][cnt]`.
4. Vì mọi `cnt ≤ 400`, chỉ cần `j ≤ 400`; chuyển `dp[i][j] = Σ_{t=0}^{i} dp[i−1][j−t]` bằng tổng tiền tố ⇒ `O(n · C)`.
5. Kiểm chứng bằng duyệt mọi hoán vị.

## 2. Đề bài & ví dụ

Cho `n` và danh sách `requirements[i] = [end_i, cnt_i]`. Đếm số hoán vị `perm` của `[0, 1, …, n−1]` sao cho với mọi yêu cầu, tiền tố `perm[0..end_i]` có **đúng** `cnt_i` nghịch thế (cặp `i < j`, `perm[i] > perm[j]`). Trả kết quả mod `10^9 + 7`.

**Ví dụ 1:** `n = 3`, `requirements = [[2,2],[0,0]]` → `2` (`[2,0,1]`, `[1,2,0]`).

**Ví dụ 2:** `n = 3`, `requirements = [[2,2],[1,1],[0,0]]` → `1`.

**Ví dụ 3:** `n = 2`, `requirements = [[0,0],[1,0]]` → `1`.

**Ràng buộc:** `2 ≤ n ≤ 300`; `1 ≤ requirements.length ≤ n`; `0 ≤ end_i ≤ n − 1`; `0 ≤ cnt_i ≤ 400`; luôn có yêu cầu với `end_i = n − 1`; các `end_i` khác nhau.

In [1]:
from typing import List
import itertools, random, time

MOD = 10 ** 9 + 7
examples = [(3, [[2, 2], [0, 0]], 2), (3, [[2, 2], [1, 1], [0, 0]], 1), (2, [[0, 0], [1, 0]], 1)]
for n, req, exp in examples:
    print('n =', n, 'req =', req, '-> kỳ vọng', exp)

n = 3 req = [[2, 2], [0, 0]] -> kỳ vọng 2
n = 3 req = [[2, 2], [1, 1], [0, 0]] -> kỳ vọng 1
n = 2 req = [[0, 0], [1, 0]] -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1. Mã Lehmer: hoán vị ↔ dãy `(t_0, t_1, …, t_{n−1})` với `0 ≤ t_i ≤ i`

Với mỗi vị trí `i`, đặt `t_i` = số phần tử **đứng trước** `perm[i]` mà **lớn hơn** `perm[i]`. Khi đó số nghịch thế của tiền tố `perm[0..e]` là `t_0 + t_1 + … + t_e`. Ánh xạ hoán vị → `(t_i)` là song ánh (có `1 · 2 · … · n = n!` dãy như vậy).

In [2]:
def lehmer_left(p):
    return [sum(1 for j in range(i) if p[j] > p[i]) for i in range(len(p))]

def inversions(a):
    return sum(1 for i in range(len(a)) for j in range(i + 1, len(a)) if a[i] > a[j])

for p in [(2, 0, 1), (1, 2, 0), (3, 1, 0, 2)]:
    t = lehmer_left(p)
    print(p, '-> t =', t, '| nghịch thế tiền tố:', [inversions(p[:e + 1]) for e in range(len(p))],
          '| tổng dồn t:', list(itertools.accumulate(t)))
codes = {tuple(lehmer_left(p)) for p in itertools.permutations(range(5))}
print('số mã Lehmer khác nhau của S5:', len(codes), '= 5! =', 120)

(2, 0, 1) -> t = [0, 1, 1] | nghịch thế tiền tố: [0, 1, 2] | tổng dồn t: [0, 1, 2]
(1, 2, 0) -> t = [0, 0, 2] | nghịch thế tiền tố: [0, 0, 2] | tổng dồn t: [0, 0, 2]
(3, 1, 0, 2) -> t = [0, 1, 2, 1] | nghịch thế tiền tố: [0, 1, 3, 4] | tổng dồn t: [0, 1, 3, 4]
số mã Lehmer khác nhau của S5: 120 = 5! = 120


### 3.2. Hệ quả: bài toán thành đếm dãy `t`

Đếm số dãy `0 ≤ t_i ≤ i` sao cho tổng dồn `T_e = t_0 + … + t_e` bằng `cnt` tại mọi `e = end`. Đây là DP "đếm số cách" kinh điển (LC 629 — K Inverse Pairs Array) có thêm các **điểm kiểm tra**.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Đi từ trái sang phải; trạng thái chỉ cần **số nghịch thế hiện tại** `j`. Ở bước `i` có `i + 1` lựa chọn `t_i ∈ [0, i]`. Khi gặp một yêu cầu `[i, cnt]`, xoá mọi trạng thái `j ≠ cnt`. Vì có yêu cầu ở `n − 1` với `cnt ≤ 400`, mọi trạng thái `j > 400` vô dụng (tổng dồn không giảm) ⇒ cắt ở `C = max cnt`.

Ví dụ 1: bước 0 `{0: 1}` (yêu cầu `[0, 0]` giữ nguyên) → bước 1 `{0: 1, 1: 1}` → bước 2 `{0: 1, 1: 2, 2: 2, 3: 1}` → yêu cầu `[2, 2]` ⇒ `2`.

## 5. Cách 1 — Brute Force: duyệt mọi hoán vị

In [3]:
def brute(n, requirements):
    cnt = 0
    for p in itertools.permutations(range(n)):
        t = lehmer_left(p); pref = list(itertools.accumulate(t))
        if all(pref[e] == c for e, c in requirements):
            cnt += 1
    return cnt % MOD

for n, req, exp in examples:
    print('brute:', brute(n, req), '| kỳ vọng', exp)

brute: 2 | kỳ vọng 2
brute: 1 | kỳ vọng 1
brute: 1 | kỳ vọng 1


**Độ phức tạp:** `O(n! · n²)` — chỉ cho `n ≤ 7`.

## 6. Ý tưởng thuật toán / bước đột phá

```
C = max cnt; need[e] = cnt (hoặc -1)
dp = [1, 0, 0, …]  (độ dài C+1)          # sau bước 0: 0 nghịch thế
áp need[0]
for i = 1..n-1:
    pre = tổng tiền tố của dp
    new[j] = pre[j] − pre[j − i − 1]     # Σ_{t=0}^{i} dp[j − t]
    áp need[i]: giữ new[need[i]], xoá phần còn lại
return dp[need[n-1]]
```

`O(n · C)` = `300 · 401 ≈ 1.2·10⁵`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DP trực tiếp `O(n · C · n)`

In [4]:
def dp_naive(n, requirements):
    need = [-1] * n
    for e, c in requirements:
        need[e] = c
    C = max(c for _, c in requirements)
    dp = [0] * (C + 1); dp[0] = 1
    if need[0] > 0:
        return 0
    for i in range(1, n):
        new = [0] * (C + 1)
        for j in range(C + 1):
            if dp[j]:
                for t in range(0, min(i, C - j) + 1):
                    new[j + t] = (new[j + t] + dp[j]) % MOD
        if need[i] >= 0:
            new = [v if j == need[i] else 0 for j, v in enumerate(new)]
        dp = new
    return dp[need[n - 1]] % MOD

for n, req, exp in examples:
    print('DP O(n·C·n):', dp_naive(n, req), '| kỳ vọng', exp)

DP O(n·C·n): 2 | kỳ vọng 2
DP O(n·C·n): 1 | kỳ vọng 1
DP O(n·C·n): 1 | kỳ vọng 1


### 7.2. Cách 3 — DP + tổng tiền tố `O(n · C)` (lời giải chính)

In [5]:
def dp_prefix(n, requirements):
    need = [-1] * n
    for e, c in requirements:
        need[e] = c
    C = max(c for _, c in requirements)
    if need[0] > 0:
        return 0
    dp = [0] * (C + 1); dp[0] = 1
    for i in range(1, n):
        pre = [0] * (C + 2)
        for j in range(C + 1):
            pre[j + 1] = (pre[j] + dp[j]) % MOD
        new = [(pre[j + 1] - pre[max(0, j - i)]) % MOD for j in range(C + 1)]
        if need[i] >= 0:
            c = need[i]
            new = [0] * (C + 1)
            new[c] = (pre[c + 1] - pre[max(0, c - i)]) % MOD
        dp = new
    return dp[need[n - 1]]

for n, req, exp in examples:
    print('DP + prefix:', dp_prefix(n, req), '| kỳ vọng', exp)

DP + prefix: 2 | kỳ vọng 2
DP + prefix: 1 | kỳ vọng 1
DP + prefix: 1 | kỳ vọng 1


### 7.3. Cách 4 — Chia theo đoạn giữa hai yêu cầu (nhân đa thức)

Giữa hai yêu cầu liên tiếp `(e₁, c₁)` và `(e₂, c₂)`, số cách là hệ số của `x^{c₂ − c₁}` trong `Π_{i=e₁+1}^{e₂} (1 + x + … + x^i)`. Kết quả = tích các hệ số đó (mỗi đoạn độc lập vì trạng thái bị "ghim" ở mỗi yêu cầu). Nhân đa thức bị cắt ở bậc `c₂ − c₁` cũng làm bằng tổng tiền tố.

In [6]:
def by_segments(n, requirements):
    req = sorted(requirements)
    if req[0][0] != 0:
        req = [[0, 0]] + req                     # tiền tố độ dài 1 luôn có 0 nghịch thế
    elif req[0][1] != 0:
        return 0
    ans = 1
    for (e1, c1), (e2, c2) in zip(req, req[1:]):
        D = c2 - c1
        if D < 0:
            return 0
        poly = [1] + [0] * D
        for i in range(e1 + 1, e2 + 1):
            pre = [0]
            for v in poly:
                pre.append((pre[-1] + v) % MOD)
            poly = [(pre[d + 1] - pre[max(0, d - i)]) % MOD for d in range(D + 1)]
        ans = ans * poly[D] % MOD
    return ans

for n, req, exp in examples:
    print('theo đoạn:', by_segments(n, req), '| kỳ vọng', exp)

theo đoạn: 2 | kỳ vọng 2
theo đoạn: 1 | kỳ vọng 1
theo đoạn: 1 | kỳ vọng 1


## 8. TRACE từng bước

Trace cách 3 trên `n = 4`, `requirements = [[3, 3], [1, 1]]`.

In [7]:
def trace(n, requirements):
    need = [-1] * n
    for e, c in requirements:
        need[e] = c
    C = max(c for _, c in requirements)
    dp = [0] * (C + 1); dp[0] = 1
    print(f'i = 0: dp = {dp}')
    for i in range(1, n):
        pre = [0] * (C + 2)
        for j in range(C + 1):
            pre[j + 1] = pre[j] + dp[j]
        new = [pre[j + 1] - pre[max(0, j - i)] for j in range(C + 1)]
        note = ''
        if need[i] >= 0:
            note = f'  -> yêu cầu [{i}, {need[i]}]: chỉ giữ dp[{need[i]}] = {new[need[i]]}'
            new = [v if j == need[i] else 0 for j, v in enumerate(new)]
        print(f'i = {i}: thêm t ∈ [0, {i}] -> dp = {new}{note}')
        dp = new
    print('đáp án:', dp[need[n - 1]], '| brute:', brute(n, requirements))

trace(4, [[3, 3], [1, 1]])

i = 0: dp = [1, 0, 0, 0]
i = 1: thêm t ∈ [0, 1] -> dp = [0, 1, 0, 0]  -> yêu cầu [1, 1]: chỉ giữ dp[1] = 1
i = 2: thêm t ∈ [0, 2] -> dp = [0, 1, 1, 1]
i = 3: thêm t ∈ [0, 3] -> dp = [0, 0, 0, 3]  -> yêu cầu [3, 3]: chỉ giữ dp[3] = 3
đáp án: 3 | brute: 3


## 9. Code nộp LeetCode

In [8]:
class Solution:
    def numberOfPermutations(self, n: int, requirements: List[List[int]]) -> int:
        MOD = 10 ** 9 + 7
        need = [-1] * n
        for e, c in requirements:
            need[e] = c
        if need[0] > 0:
            return 0
        C = max(c for _, c in requirements)
        dp = [0] * (C + 1)
        dp[0] = 1
        for i in range(1, n):
            pre = [0] * (C + 2)
            s = 0
            for j in range(C + 1):
                s += dp[j]
                pre[j + 1] = s
            if need[i] >= 0:
                c = need[i]
                new = [0] * (C + 1)
                new[c] = (pre[c + 1] - pre[max(0, c - i)]) % MOD
            else:
                new = [(pre[j + 1] - pre[max(0, j - i)]) % MOD for j in range(C + 1)]
            dp = new
        return dp[need[n - 1]] % MOD

for n, req, exp in examples:
    got = Solution().numberOfPermutations(n, req)
    print(got, 'OK' if got == exp else 'SAI')

2 OK
1 OK
1 OK


## 10. Vì sao thuật toán đúng

1. **Song ánh Lehmer:** hoán vị ↔ dãy `t` với `0 ≤ t_i ≤ i`, và nghịch thế của tiền tố `[0..e]` bằng `Σ_{i ≤ e} t_i` (mỗi nghịch thế `(a, b)` với `a < b ≤ e` được đếm đúng một lần ở `t_b`).
2. **DP:** `dp_i[j]` = số dãy `t_0..t_i` có tổng `j` và thoả các yêu cầu `≤ i`. Chuyển: `dp_i[j] = Σ_{t=0}^{i} dp_{i−1}[j − t]`, rồi xoá các `j ≠ need[i]` nếu có yêu cầu.
3. **Cắt ở `C`:** tổng dồn không giảm, và có yêu cầu ở `n − 1` với giá trị `≤ C` ⇒ trạng thái `j > C` không bao giờ tới đích.
4. **Tổng tiền tố:** `Σ_{t=0}^{i} dp[j − t] = pre[j+1] − pre[max(0, j − i)]` — đúng định nghĩa.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Brute | `O(n! · n²)` | `O(n)` |
| 2. DP trực tiếp | `O(n² · C)` | `O(C)` |
| 3. DP + tổng tiền tố | `O(n · C)` | `O(C)` |
| 4. Theo đoạn | `O(n · C)` | `O(C)` |

`C ≤ 400`, `n ≤ 300` ⇒ cách 3 khoảng `1.2·10⁵` phép.

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [9]:
impls = {'naive': dp_naive, 'prefix': dp_prefix, 'segments': by_segments,
         'submit': lambda n, r: Solution().numberOfPermutations(n, r)}
edge = [(2, [[1, 1]], 1), (2, [[1, 0]], 1), (3, [[2, 3]], 1), (3, [[2, 4]], 0), (4, [[3, 3]], 6),
        (3, [[0, 1], [2, 1]], 0), (4, [[1, 1], [3, 1]], 1), (5, [[4, 5]], 22)]
for n, req, exp in examples + edge:
    assert brute(n, req) == exp, (n, req, brute(n, req))
    for name, f in impls.items():
        assert f(n, req) == exp, (name, n, req)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


Giải thích: `[[0, 1], …]` — tiền tố một phần tử không thể có nghịch thế ⇒ `0`. `(3, [[2, 3]])` — chỉ hoán vị giảm dần. `(4, [[3, 3]])` — số hoán vị của 4 phần tử có 3 nghịch thế (số Mahonian) là `6`. `[[1, 1], [3, 1]]`: sau khi tiền tố 2 phần tử đã có 1 nghịch thế, hai phần tử sau phải không thêm nghịch thế nào.

### 12.2. Đối chiếu ngẫu nhiên với brute

In [10]:
rng = random.Random(3193)
cnt = 0
for _ in range(300):
    n = rng.randint(2, 6)
    ends = rng.sample(range(n - 1), rng.randint(0, n - 1)) + [n - 1]
    req = [[e, rng.randint(0, e * (e + 1) // 2 + 1)] for e in ends]
    if rng.random() < 0.5:                          # sinh yêu cầu "khả thi" từ một hoán vị thật
        p = rng.sample(range(n), n); pref = list(itertools.accumulate(lehmer_left(p)))
        req = [[e, pref[e]] for e in ends]
    rng.shuffle(req)
    exp = brute(n, req)
    for name, f in impls.items():
        assert f(n, req) == exp, (name, n, req)
    cnt += 1
print('Khớp brute trên', cnt, 'bộ ngẫu nhiên.')

Khớp brute trên 300 bộ ngẫu nhiên.


### 12.3. Ca lớn `n = 300`, `cnt` tới 400

In [11]:
big = {
    'chỉ yêu cầu cuối': (300, [[299, 400]]),
    'nhiều yêu cầu': (300, [[e, min(400, e)] for e in range(0, 300, 3)] + [[299, 400]]),
    'mọi vị trí': (300, [[e, min(400, e * (e + 1) // 2)] for e in range(300)]),
}
for name, (n, req) in big.items():
    t = time.time(); x = Solution().numberOfPermutations(n, req); t1 = time.time() - t
    y = by_segments(n, req)
    assert x == y
    print(f'{name:16s}: {x:10d} | nộp {t1:.3f}s')

chỉ yêu cầu cuối:  615671924 | nộp 0.020s
nhiều yêu cầu   :  214812102 | nộp 0.014s
mọi vị trí      :          1 | nộp 0.005s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ áp yêu cầu ở vị trí cuối (bỏ qua các tiền tố khác)

In [12]:
def wrong_only_last(n, requirements):
    c = dict(map(tuple, requirements))[n - 1]
    return Solution().numberOfPermutations(n, [[n - 1, c]])          # SAI

print('sai :', wrong_only_last(*examples[1][:2]), '| đúng:', Solution().numberOfPermutations(*examples[1][:2]))

sai : 2 | đúng: 1


**Vì sao sai:** yêu cầu `[1, 1]` loại bỏ `[1, 2, 0]`; bỏ qua nó thì đếm dư.

### Lỗi 2 — Lệch chỉ số: áp yêu cầu `end` sau bước `end + 1`

In [13]:
def wrong_off_by_one(n, requirements):
    need = [-1] * (n + 1)
    for e, c in requirements:
        need[e + 1] = c                                                 # SAI: coi end là độ dài
    C = max(c for _, c in requirements)
    dp = [0] * (C + 1); dp[0] = 1
    for i in range(1, n):
        new = [sum(dp[max(0, j - i): j + 1]) % MOD for j in range(C + 1)]
        if need[i] >= 0:
            new = [v if j == need[i] else 0 for j, v in enumerate(new)]
        dp = new
    last = need[n] if need[n] >= 0 else need[n - 1]
    return dp[last]

args = (3, [[1, 0], [2, 1]])
print('sai :', wrong_off_by_one(*args), '| đúng:', Solution().numberOfPermutations(*args))

sai : 0 | đúng: 1


**Vì sao sai:** `end` là **chỉ số** cuối của tiền tố (độ dài `end + 1`). Lệch một bước làm yêu cầu áp vào tiền tố sai.

### Lỗi 3 — Quên kiểm yêu cầu `end = 0` với `cnt > 0`

In [14]:
def wrong_skip_first(n, requirements):
    need = [-1] * n
    for e, c in requirements:
        need[e] = c
    C = max(c for _, c in requirements)
    dp = [0] * (C + 1); dp[0] = 1                                       # SAI: không xét need[0]
    for i in range(1, n):
        new = [sum(dp[max(0, j - i): j + 1]) % MOD for j in range(C + 1)]
        if need[i] >= 0:
            new = [v if j == need[i] else 0 for j, v in enumerate(new)]
        dp = new
    return dp[need[n - 1]]

args = (3, [[0, 1], [2, 1]])
print('sai :', wrong_skip_first(*args), '| đúng:', Solution().numberOfPermutations(*args))

sai : 2 | đúng: 0


**Vì sao sai:** tiền tố một phần tử có 0 nghịch thế; yêu cầu `[0, 1]` làm đáp án bằng 0. Vòng lặp bắt đầu từ `i = 1` nên dễ bỏ sót kiểm tra ở `i = 0`.

### Lỗi 4 — Cho phép thêm `0..i−1` nghịch thế (thiếu một lựa chọn)

In [15]:
def wrong_range(n, requirements):
    need = [-1] * n
    for e, c in requirements:
        need[e] = c
    if need[0] > 0:
        return 0
    C = max(c for _, c in requirements)
    dp = [0] * (C + 1); dp[0] = 1
    for i in range(1, n):
        new = [sum(dp[max(0, j - i + 1): j + 1]) % MOD for j in range(C + 1)]   # SAI: t ∈ [0, i-1]
        if need[i] >= 0:
            new = [v if j == need[i] else 0 for j, v in enumerate(new)]
        dp = new
    return dp[need[n - 1]]

print('sai :', wrong_range(*examples[0][:2]), '| đúng:', Solution().numberOfPermutations(*examples[0][:2]))

sai : 0 | đúng: 2


**Vì sao sai:** phần tử thứ `i` có `i + 1` vị trí tương đối (nhỏ nhất … lớn nhất) ⇒ tạo `0..i` nghịch thế. Thiếu `t = i` (phần tử mới nhỏ nhất) làm mất các hoán vị.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đếm hoán vị theo số nghịch thế" ⇒ mã Lehmer, DP `0 ≤ t_i ≤ i` (LC 629).
- Ràng buộc trên **tiền tố** ⇒ lọc trạng thái tại các điểm kiểm tra.
- Chuyển trạng thái là tổng một **cửa sổ** liên tiếp ⇒ tổng tiền tố.
- Giá trị yêu cầu nhỏ (`≤ 400`) ⇒ cắt chiều trạng thái.

## 15. Biến thể của bài

1. **LC 629 K Inverse Pairs Array:** một yêu cầu duy nhất ở cuối.
2. Yêu cầu dạng "**không quá** `cnt`" ⇒ giữ các `j ≤ cnt` thay vì đúng `cnt`.
3. Yêu cầu trên **hậu tố** ⇒ dùng mã Lehmer phải (`t_i` = số phần tử sau nhỏ hơn).
4. Đếm số nghịch thế **kỳ vọng** của hoán vị ngẫu nhiên thoả ràng buộc.

In [16]:
# Biến thể 2: yêu cầu "tiền tố [0..end] có NHIỀU NHẤT cnt nghịch thế"
def at_most(n, requirements):
    need = [-1] * n
    for e, c in requirements:
        need[e] = c
    C = max(c for _, c in requirements)
    dp = [0] * (C + 1); dp[0] = 1
    for i in range(1, n):
        pre = [0]
        for v in dp:
            pre.append((pre[-1] + v) % MOD)
        dp = [(pre[j + 1] - pre[max(0, j - i)]) % MOD for j in range(C + 1)]
        if need[i] >= 0:
            dp = [v if j <= need[i] else 0 for j, v in enumerate(dp)]
    return sum(dp[:need[n - 1] + 1]) % MOD

def brute_at_most(n, requirements):
    cnt = 0
    for p in itertools.permutations(range(n)):
        pref = list(itertools.accumulate(lehmer_left(p)))
        cnt += all(pref[e] <= c for e, c in requirements)
    return cnt

for _ in range(100):
    n = rng.randint(2, 6); ends = rng.sample(range(1, n - 1), rng.randint(0, max(0, n - 2))) + [n - 1]
    req = [[e, rng.randint(0, e * (e + 1) // 2)] for e in ends]
    assert at_most(n, req) == brute_at_most(n, req)
print('Biến thể "nhiều nhất cnt": khớp brute.')

Biến thể "nhiều nhất cnt": khớp brute.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Duyệt hoán vị | `O(n!·n²)` | Oracle |
| 2 | DP trực tiếp | `O(n²C)` | Dễ viết, `3.6·10⁷` — chậm |
| 3 | DP + tổng tiền tố | `O(nC)` | **Nộp bài** |
| 4 | Theo đoạn (nhân đa thức) | `O(nC)` | Nhìn rõ tính độc lập giữa các đoạn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `n = 4`, `requirements = [[3, 0]]`.

**Bài 2 (tính tay):** `n = 4`, `requirements = [[3, 2]]` (số hoán vị của 4 phần tử có 2 nghịch thế).

**Bài 3 (điền TODO):** dựng **một** hoán vị thoả mãn (hoặc `None`) bằng truy vết DP và giải mã Lehmer.

**Bài 4 (tự viết lại):** LC 629 — số hoán vị của `n` phần tử có đúng `k` nghịch thế, `n, k ≤ 1000`.

**Bài 5 (mini challenge):** tìm hoán vị thoả mãn **nhỏ nhất theo thứ tự từ điển**.

In [17]:
def construct_one_todo(n, requirements):
    # TODO
    raise NotImplementedError

def k_inverse_pairs_todo(n, k):
    # TODO
    raise NotImplementedError

def lexicographically_smallest_todo(n, requirements):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) 0 nghịch thế ⇒ tăng dần. (2) Chỉ một hoán vị. (3) `1`.

**Bài 2.** (1) Hệ số `x²` trong `(1)(1+x)(1+x+x²)(1+x+x²+x³)`. (2) `= 5`. (3) Kiểm bằng `brute`.

**Bài 3.** (1) Lưu các bảng `dp_i` (sau khi lọc). (2) Truy ngược từ `j = need[n−1]`: chọn `t_i` sao cho `dp_{i−1}[j − t_i] > 0`. (3) Giải mã: duyệt `i` từ phải sang trái, phần tử thứ `i` là phần tử lớn thứ `t_i + 1` trong tập còn lại của **tiền tố** — dễ hơn: dựng thứ tự tương đối bằng chèn.

**Bài 4.** (1) Chính là cách 3 với một yêu cầu `[n−1, k]`. (2) `C = k`. (3) Đối chiếu với `brute`.

**Bài 5.** (1) Chọn `perm[0], perm[1], …` lần lượt từ nhỏ đến lớn. (2) Mỗi lựa chọn cần biết "còn hoàn thành được không" — khó hơn vì `t_i` phụ thuộc **giá trị** tuyệt đối, không chỉ thứ tự tương đối. (3) Với `n` nhỏ: thử tham lam + kiểm tra khả thi bằng duyệt; tổng quát dùng mã Lehmer **phải** (đếm phần tử nhỏ hơn đứng sau).

## 19. Đáp án tham khảo cho bài tập

In [18]:
# Bài 1, 2, 4
assert Solution().numberOfPermutations(4, [[3, 0]]) == 1
assert Solution().numberOfPermutations(4, [[3, 2]]) == 5

def k_inverse_pairs(n, k):
    return Solution().numberOfPermutations(n, [[n - 1, k]]) if n >= 2 else int(k == 0)

for n in range(1, 7):
    for k in range(0, n * (n - 1) // 2 + 2):
        assert k_inverse_pairs(n, k) == sum(1 for p in itertools.permutations(range(n)) if inversions(p) == k)
t = time.time(); v = k_inverse_pairs(1000, 1000); print('LC 629 n = k = 1000:', v, f'({time.time() - t:.2f}s)')
print('Bài 1: 1 | Bài 2: 5 | Bài 4: khớp đếm trực tiếp với n ≤ 6')

LC 629 n = k = 1000: 663677020 (0.18s)
Bài 1: 1 | Bài 2: 5 | Bài 4: khớp đếm trực tiếp với n ≤ 6


In [19]:
# Bài 3, 5
def construct_one(n, requirements):
    need = [-1] * n
    for e, c in requirements:
        need[e] = c
    if need[0] > 0:
        return None
    C = max(c for _, c in requirements)
    tables = [[1] + [0] * C]
    for i in range(1, n):
        dp = tables[-1]
        new = [sum(dp[max(0, j - i): j + 1]) for j in range(C + 1)]
        if need[i] >= 0:
            new = [v if j == need[i] else 0 for j, v in enumerate(new)]
        tables.append(new)
    j = need[n - 1]
    if tables[-1][j] == 0:
        return None
    ts = [0] * n
    for i in range(n - 1, 0, -1):
        for t_ in range(0, min(i, j) + 1):
            if tables[i - 1][j - t_]:
                ts[i] = t_; j -= t_; break
    order = []                                         # chèn: phần tử thứ i có t_i phần tử lớn hơn đứng trước
    for i in range(n):
        order.insert(len(order) - ts[i], i)            # order: thứ hạng tăng dần của các vị trí
    perm = [0] * n
    for rank, pos in enumerate(order):
        perm[pos] = rank
    return perm

def ok(p, requirements):
    return all(inversions(p[:e + 1]) == c for e, c in requirements)

def lexicographically_smallest(n, requirements):
    for p in itertools.permutations(range(n)):         # sinh theo thứ tự từ điển (n nhỏ)
        if ok(p, requirements):
            return list(p)
    return None

for _ in range(250):
    n = rng.randint(2, 6); ends = rng.sample(range(n - 1), rng.randint(0, n - 1)) + [n - 1]
    p0 = rng.sample(range(n), n)
    req = [[e, inversions(p0[:e + 1]) + (rng.random() < 0.15)] for e in ends]
    p = construct_one(n, req); cntv = Solution().numberOfPermutations(n, req)
    assert (p is None) == (cntv == 0)
    if p is not None:
        assert sorted(p) == list(range(n)) and ok(p, req)
    q = lexicographically_smallest(n, req)
    assert (q is None) == (cntv == 0)
print('Bài 3: dựng được hoán vị hợp lệ khi đáp án > 0 | Bài 5 (n nhỏ):',
      lexicographically_smallest(3, [[2, 2], [0, 0]]), construct_one(3, [[2, 2], [0, 0]]))

Bài 3: dựng được hoán vị hợp lệ khi đáp án > 0 | Bài 5 (n nhỏ): [1, 2, 0] [2, 0, 1]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
need[e] = cnt; C = max cnt; if need[0] > 0: return 0
dp = [1] + [0] * C
for i in 1..n-1:
    pre = prefix(dp)
    dp[j] = pre[j+1] - pre[max(0, j-i)]      # t ∈ [0, i]
    if need[i] >= 0: giữ dp[need[i]]
return dp[need[n-1]]
```

**Checklist:**

- [ ] `t_i ∈ [0, i]` (đủ `i + 1` lựa chọn)?
- [ ] `end` là **chỉ số** (độ dài `end + 1`)?
- [ ] Kiểm yêu cầu ở `end = 0`?
- [ ] Cắt chiều trạng thái ở `max cnt`?
- [ ] Mod sau mỗi phép trừ tổng tiền tố (`% MOD` cho số âm)?